In [6]:
!pip install simpy

In this problem you, can simulate a simplified airport security system at a busy airport. Passengers arrive
according to a Poisson distribution with λ1 = 5 per minute (i.e., mean interarrival rate 1 = 0.2 minutes)
to the ID/boarding-pass check queue, where there are several servers who each have exponential service
time with mean rate 2 = 0.75 minutes. [Hint: model them as one block that has more than one resource.]
After that, the passengers are assigned to the shortest of the several personal-check queues, where they go
through the personal scanner (time is uniformly distributed between 0.5 minutes and 1 minute).
Use the Arena software (PC users) or Python with SimPy (PC or Mac users) to build a simulation of the
system, and then vary the number of ID/boarding-pass checkers and personal-check queues to determine
how many are needed to keep average wait times below 15 minutes. [If you’re using SimPy, or if you
have access to a non-student version of Arena, you can use λ1 = 50 to simulate a busier airport.]


In [23]:
# Importing packages
import simpy
import random
import pandas as pd
from statistics import mean

In [29]:
# Simulation parameters
RANDOM_SEED = 42
SIM_TIME = 100  # minutes
LAMBDA_ARRIVAL = 50  # high-traffic arrival rate
MEAN_ID_TIME = 0.75  # average time at ID check
MIN_SCAN_TIME = 0.5
MAX_SCAN_TIME = 1.0

random.seed(RANDOM_SEED)

def passenger(env, name, id_checkers, scanners, wait_times):
    arrival_time = env.now

    with id_checkers.request() as request:
        yield request
        yield env.timeout(random.expovariate(1.0 / MEAN_ID_TIME))

    shortest_scanner = min(scanners, key=lambda s: len(s.queue))
    with shortest_scanner.request() as request:
        yield request
        yield env.timeout(random.uniform(MIN_SCAN_TIME, MAX_SCAN_TIME))

    wait_times.append(env.now - arrival_time)

def arrival_process(env, id_checkers, scanners, wait_times):
    i = 0
    while True:
        yield env.timeout(random.expovariate(LAMBDA_ARRIVAL))
        i += 1
        env.process(passenger(env, f'Passenger {i}', id_checkers, scanners, wait_times))

def run_simulation(n_id_checkers=5, n_scanners=5):
    wait_times = []
    env = simpy.Environment()
    id_checkers = simpy.Resource(env, capacity=n_id_checkers)
    scanners = [simpy.Resource(env, capacity=1) for _ in range(n_scanners)]
    env.process(arrival_process(env, id_checkers, scanners, wait_times))
    env.run(until=SIM_TIME)
    return mean(wait_times) if wait_times else 0

# Try more combinations: ID checkers and scanners from 5 to 20
results = []
for id_servers in range(20, 30):
    for n_scanners in range(20, 30):
        avg_wait = run_simulation(n_id_checkers=id_servers, n_scanners=n_scanners)
        if avg_wait < 15:
            results.append((id_servers, n_scanners, avg_wait))
            print(f"ID Checkers: {id_servers}, Scanners: {n_scanners}, Avg Wait: {avg_wait:.2f} min")

# Export results to DataFrame (optional)
df = pd.DataFrame(results, columns=["ID Checkers", "Scanners", "Avg Wait Time (min)"])
print("\nConfigurations meeting target wait time (<15 min):")
print(df)

ID Checkers: 27, Scanners: 28, Avg Wait: 13.86 min
ID Checkers: 27, Scanners: 29, Avg Wait: 13.75 min
ID Checkers: 28, Scanners: 27, Avg Wait: 14.62 min
ID Checkers: 28, Scanners: 28, Avg Wait: 14.48 min
ID Checkers: 28, Scanners: 29, Avg Wait: 14.04 min
ID Checkers: 29, Scanners: 28, Avg Wait: 14.06 min
ID Checkers: 29, Scanners: 29, Avg Wait: 14.04 min

Configurations meeting target wait time (<15 min):
   ID Checkers  Scanners  Avg Wait Time (min)
0           27        28            13.863847
1           27        29            13.753932
2           28        27            14.622368
3           28        28            14.482536
4           28        29            14.035586
5           29        28            14.061878
6           29        29            14.038064


In [27]:
# Simulation parameters
RANDOM_SEED = 42
SIM_TIME = 100  # minutes
LAMBDA_ARRIVAL = 30  # high-traffic arrival rate
MEAN_ID_TIME = 0.75  # average time at ID check
MIN_SCAN_TIME = 0.5
MAX_SCAN_TIME = 1.0

random.seed(RANDOM_SEED)

def passenger(env, name, id_checkers, scanners, wait_times):
    arrival_time = env.now

    with id_checkers.request() as request:
        yield request
        yield env.timeout(random.expovariate(1.0 / MEAN_ID_TIME))

    shortest_scanner = min(scanners, key=lambda s: len(s.queue))
    with shortest_scanner.request() as request:
        yield request
        yield env.timeout(random.uniform(MIN_SCAN_TIME, MAX_SCAN_TIME))

    wait_times.append(env.now - arrival_time)

def arrival_process(env, id_checkers, scanners, wait_times):
    i = 0
    while True:
        yield env.timeout(random.expovariate(LAMBDA_ARRIVAL))
        i += 1
        env.process(passenger(env, f'Passenger {i}', id_checkers, scanners, wait_times))

def run_simulation(n_id_checkers=5, n_scanners=5):
    wait_times = []
    env = simpy.Environment()
    id_checkers = simpy.Resource(env, capacity=n_id_checkers)
    scanners = [simpy.Resource(env, capacity=1) for _ in range(n_scanners)]
    env.process(arrival_process(env, id_checkers, scanners, wait_times))
    env.run(until=SIM_TIME)
    return mean(wait_times) if wait_times else 0

# Try more combinations: ID checkers and scanners from 5 to 20
results = []
for id_servers in range(5, 21):
    for n_scanners in range(5, 21):
        avg_wait = run_simulation(n_id_checkers=id_servers, n_scanners=n_scanners)
        if avg_wait < 15:
            results.append((id_servers, n_scanners, avg_wait))
            print(f"ID Checkers: {id_servers}, Scanners: {n_scanners}, Avg Wait: {avg_wait:.2f} min")

# Export results to DataFrame (optional)
df = pd.DataFrame(results, columns=["ID Checkers", "Scanners", "Avg Wait Time (min)"])
print("\nConfigurations meeting target wait time (<15 min):")
print(df)

ID Checkers: 16, Scanners: 19, Avg Wait: 13.94 min
ID Checkers: 16, Scanners: 20, Avg Wait: 14.65 min
ID Checkers: 17, Scanners: 16, Avg Wait: 14.24 min
ID Checkers: 17, Scanners: 18, Avg Wait: 14.23 min
ID Checkers: 17, Scanners: 19, Avg Wait: 13.69 min
ID Checkers: 17, Scanners: 20, Avg Wait: 13.82 min
ID Checkers: 18, Scanners: 16, Avg Wait: 14.60 min
ID Checkers: 18, Scanners: 17, Avg Wait: 14.93 min
ID Checkers: 18, Scanners: 18, Avg Wait: 12.59 min
ID Checkers: 18, Scanners: 19, Avg Wait: 11.39 min
ID Checkers: 18, Scanners: 20, Avg Wait: 11.63 min
ID Checkers: 19, Scanners: 17, Avg Wait: 13.33 min
ID Checkers: 19, Scanners: 18, Avg Wait: 13.44 min
ID Checkers: 19, Scanners: 19, Avg Wait: 10.97 min
ID Checkers: 19, Scanners: 20, Avg Wait: 10.37 min
ID Checkers: 20, Scanners: 17, Avg Wait: 14.39 min
ID Checkers: 20, Scanners: 18, Avg Wait: 11.55 min
ID Checkers: 20, Scanners: 19, Avg Wait: 10.46 min
ID Checkers: 20, Scanners: 20, Avg Wait: 7.30 min

Configurations meeting target w